# Notebook Security and Usage Guardrails
- Do not store secrets, passwords, tokens, or keys in notebook source or outputs.
- Load credentials only from environment variables (`.env`) and keep outputs cleared before commit.
- Canonical production logic lives under `data_pipeline/src`; this notebook is for exploration/reference.


In [3]:
import os
from pathlib import Path

REQUIRED_ENV_VARS = ["BGG_USERNAME", "BGG_PASSWORD"]
missing = [name for name in REQUIRED_ENV_VARS if not os.getenv(name)]
if missing:
    raise ValueError(f"Missing required environment variable(s): {', '.join(missing)}")

project_root = Path.cwd()
if not (project_root / "data").exists():
    raise ValueError("Expected ./data directory at repository root; run notebook from repo root.")

print("Environment guard check passed.")


ValueError: Missing required environment variable(s): BGG_USERNAME, BGG_PASSWORD

In [ ]:
%config Completer.use_jedi = False
%load_ext autoreload

# Imports and Loads

In [1]:
from bs4 import BeautifulSoup
import os
import requests
from zipfile import ZipFile
from io import BytesIO
from datetime import datetime
import pandas as pd
import logging
import csv
from pathlib import Path
from time import sleep

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from dotenv import load_dotenv, find_dotenv

In [2]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("../../logs/get_ranks.log"), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

# Functions

In [5]:
def get_driver_and_cookies():
    """
    Initialize Selenium WebDriver and authenticate with BoardGameGeek.
    Returns authentication cookies needed for subsequent API requests.

    Returns:
        dict: Authentication cookies for BGG API requests
    """
    logger.info("Initializing web driver and getting cookies")
    LOGIN_USERNAME_FIELD = '//*[@id="inputUsername"]'
    LOGIN_PASSWORD_FIELD = '//*[@id="inputPassword"]'
    LOGIN_BUTTON = '//*[@id="mainbody"]/div/div/gg-login-page/div[1]/div/gg-login-form/form/fieldset/div[3]/button[1]'

    load_dotenv(find_dotenv())
    USERNAME = os.getenv("BGG_USERNAME")
    PASSWORD = os.getenv("BGG_PASSWORD")
    if not USERNAME or not PASSWORD:
        logger.error("Missing BGG_USERNAME or BGG_PASSWORD in environment")
        raise ValueError("Missing BGG_USERNAME or BGG_PASSWORD in environment")

    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    cookies = {}

    driver = webdriver.Chrome(
        service=Service("/usr/lib/chromium-browser/chromedriver"),
        options=chrome_options,
    )
    logger.info("Chrome driver initialized successfully")

    driver.get("https://boardgamegeek.com/login")
    login = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, LOGIN_USERNAME_FIELD))
    )
    password = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, LOGIN_PASSWORD_FIELD))
    )

    login_button = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, LOGIN_BUTTON))
    )

    login.send_keys(USERNAME)
    password.send_keys(PASSWORD)

    login_button.click()
    sleep(1)
    logger.info("Successfully logged in to BoardGameGeek")

    selenium_cookies = driver.get_cookies()
    for cookie in selenium_cookies:
        cookies[cookie["name"]] = cookie["value"]
    logger.info("Successfully retrieved cookies")
    return cookies

# Code

In [6]:
cookies = get_driver_and_cookies()

2026-03-13 12:16:01,187 - __main__ - INFO - Initializing web driver and getting cookies
2026-03-13 12:16:06,567 - __main__ - INFO - Chrome driver initialized successfully


TimeoutException: Message: 
Stacktrace:
#0 0x5cc404778d7a <unknown>
#1 0x5cc4041ac722 <unknown>
#2 0x5cc404200dde <unknown>
#3 0x5cc404201101 <unknown>
#4 0x5cc40424c084 <unknown>
#5 0x5cc4042492a5 <unknown>
#6 0x5cc4041f2e64 <unknown>
#7 0x5cc4041f3c21 <unknown>
#8 0x5cc40473d31f <unknown>
#9 0x5cc40474030e <unknown>
#10 0x5cc40472b37e <unknown>
#11 0x5cc404740aa9 <unknown>
#12 0x5cc4047139f7 <unknown>
#13 0x5cc404765d98 <unknown>
#14 0x5cc404765fc3 <unknown>
#15 0x5cc404777328 <unknown>
#16 0x784c8609caa4 <unknown>
#17 0x784c86129c6c <unknown>


In [7]:
from selenium.common.exceptions import TimeoutException

def _find_first(driver, selectors, timeout=30):
    wait = WebDriverWait(driver, timeout)
    last_exc = None
    for by, sel in selectors:
        try:
            return wait.until(EC.presence_of_element_located((by, sel)))
        except Exception as exc:
            last_exc = exc
    raise last_exc if last_exc else TimeoutException("No selector matched")

In [9]:
load_dotenv(find_dotenv())
USERNAME = os.getenv("BGG_USERNAME")
PASSWORD = os.getenv("BGG_PASSWORD")
if not USERNAME or not PASSWORD:
    logger.error("Missing BGG_USERNAME or BGG_PASSWORD in environment")
    raise ValueError("Missing BGG_USERNAME or BGG_PASSWORD in environment")

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
cookies = {}
driver = webdriver.Chrome(
    service=Service("/usr/lib/chromium-browser/chromedriver"),
    options=chrome_options,
)
logger.info("Chrome driver initialized successfully")

2026-03-13 12:21:36,104 - __main__ - INFO - Chrome driver initialized successfully


In [10]:
driver.get("https://boardgamegeek.com/login")
WebDriverWait(driver, 30).until(lambda d: d.execute_script("return document.readyState") == "complete")

True

In [11]:
username_el = _find_first(driver, [
    (By.CSS_SELECTOR, "input#inputUsername"),
    (By.CSS_SELECTOR, "input[name='username']"),
    (By.CSS_SELECTOR, "input[autocomplete='username']"),
])

TimeoutException: Message: 
Stacktrace:
#0 0x600c385ced7a <unknown>
#1 0x600c38002722 <unknown>
#2 0x600c38056dde <unknown>
#3 0x600c38057101 <unknown>
#4 0x600c380a2084 <unknown>
#5 0x600c3809f2a5 <unknown>
#6 0x600c38048e64 <unknown>
#7 0x600c38049c21 <unknown>
#8 0x600c3859331f <unknown>
#9 0x600c3859630e <unknown>
#10 0x600c3858137e <unknown>
#11 0x600c38596aa9 <unknown>
#12 0x600c385699f7 <unknown>
#13 0x600c385bbd98 <unknown>
#14 0x600c385bbfc3 <unknown>
#15 0x600c385cd328 <unknown>
#16 0x72ec8ee9caa4 <unknown>
#17 0x72ec8ef29c6c <unknown>


In [ ]:







password_el = _find_first(driver, [
    (By.CSS_SELECTOR, "input#inputPassword"),
    (By.CSS_SELECTOR, "input[name='password']"),
    (By.CSS_SELECTOR, "input[type='password']"),
])

submit_el = _find_first(driver, [
    (By.CSS_SELECTOR, "button[type='submit']"),
    (By.CSS_SELECTOR, "button[class*='login']"),
])

username_el.clear()
username_el.send_keys(USERNAME)
password_el.clear()
password_el.send_keys(PASSWORD)
submit_el.click()


In [ ]:
logger.info("Fetching boardgame ranks")
bg_ranks_pg_url = "https://boardgamegeek.com/data_dumps/bg_ranks"
resp = requests.get(bg_ranks_pg_url, cookies=cookies)
soup = BeautifulSoup(resp.content, "html.parser")
bg_ranks_url = soup.find("div", {"id": "maincontent"})("a")[0]["href"]
bg_ranks_zip = requests.get(bg_ranks_url)
queried_at_utc = datetime.now().replace(microsecond=0).isoformat()

In [ ]:
save_file = True
with ZipFile(BytesIO(bg_ranks_zip.content)) as archive:
    with archive.open("boardgames_ranks.csv") as csv_file:
        df_bg_ranks = pd.read_csv(csv_file)
        df_bg_ranks["name"] = df_bg_ranks["name"].str.replace("[“”]", '"', regex=True)
        df_bg_ranks["queried_at_utc"] = queried_at_utc
        logger.info(f"Successfully loaded {len(df_bg_ranks)} boardgames")
        if save_file:
            # Create data directory if it doesn't exist
            data_dir = Path(__file__).parent.parent.parent / "data" / "crawler"
            data_dir.mkdir(parents=True, exist_ok=True)
            
            output_file = data_dir / f"boardgame_ranks_{queried_at_utc[:10].replace('-','')}.csv"
            df_bg_ranks.to_csv(
                output_file,
                index=False,
                sep="|",
                escapechar="\\",
                quoting=csv.QUOTE_NONE,
            )
            logger.info(f"Saved rankings to {output_file}")

In [ ]:
df_bg_ranks.head()